# Phantasm GPU training

This notebook runs exactly the same packaged command as headless training
(`phantasm train`), with the same defaults: leakage-safe splits produced by
`phantasm format`, response-only loss, validation-driven best-checkpoint
selection, and a saved `training_config.json`.

Open it from a Phantasm checkout on a Linux x86_64 NVIDIA GPU runtime and read
`docs/TRAINING.md` first; the pinned environment needs a real GPU smoke test on
your machine.

## 1. Install the pinned training environment

In [ ]:
import subprocess
from pathlib import Path

repo = Path.cwd()
assert (repo / "src/phantasm/resources/training.txt").is_file(), (
    "Change directory to your Phantasm checkout"
)
subprocess.run(["uv", "venv", "--python", "3.11", ".venv-training"], check=True)
python = str(repo / ".venv-training/bin/python")
subprocess.run(
    [
        "uv",
        "pip",
        "sync",
        "--python",
        python,
        "--require-hashes",
        "src/phantasm/resources/training.txt",
    ],
    check=True,
)
subprocess.run(["uv", "pip", "install", "--python", python, "--no-deps", "-e", "."], check=True)

## 2. Build leakage-safe datasets

Whole conversation sessions are assigned to train/validation/test before any
sliding window is generated, so no source message reaches two splits. Skip this
cell if you already copied `dataset_*_sharegpt.jsonl` into the runtime.

In [ ]:
subprocess.run(
    [
        python,
        "-m",
        "phantasm.cli",
        "format",
        "-i",
        "parsed.json",
        "-p",
        "dataset",
        "--val-ratio",
        "0.075",
        "--test-ratio",
        "0.075",
    ],
    check=True,
)

## 3. Check the data before spending GPU time

`inspect` reports how much of the data is actually the target speaking;
`audit` is a local heuristic privacy scan that redacts whatever it matches.

In [ ]:
for command in (["inspect", "parsed.json"], ["audit", "dataset_train_sharegpt.jsonl"]):
    subprocess.run([python, "-m", "phantasm.cli", *command], check=True)

## 4. Train

The dry run validates the files, prints the effective configuration and its
SHA-256 fingerprints, and loads no model dependencies.

In [ ]:
command = [
    python,
    "-m",
    "phantasm.cli",
    "train",
    "--train-dataset",
    "dataset_train_sharegpt.jsonl",
    "--eval-dataset",
    "dataset_val_sharegpt.jsonl",
    "--loss",
    "response_only",
    "--max-steps",
    "120",
    "--eval-steps",
    "10",
    "--early-stopping-patience",
    "3",
    "--output-dir",
    "phantasm_model",
    "--quant-method",
    "q4_k_m",
]
subprocess.run(command + ["--dry-run"], check=True)

In [ ]:
subprocess.run(command, check=True)

## 5. Inspect the result

`run.json` records package versions, dataset hashes, the derived loss markers,
metrics and the selected checkpoint. `training_config.json` holds the effective
configuration on its own, so a later run can be reproduced from it.

In [ ]:
import json

print(json.dumps(json.loads(Path("phantasm_model/run.json").read_text()), indent=2))

## 6. Measure persona fidelity

Score the exported GGUF against the held-out test split. Supply
`--baseline-model` with the un-tuned base GGUF to quantify what fine-tuning
changed. This needs `llama-cpp-python`, which is not part of the training lock;
running it in your local inference environment is usually easier.

In [ ]:
# subprocess.run(
#     [
#         python,
#         "-m",
#         "phantasm.cli",
#         "evaluate",
#         "dataset_test_sharegpt.jsonl",
#         "--model",
#         "phantasm_model/gguf/MODEL.gguf",
#         "-o",
#         "persona_evaluation.json",
#     ],
#     check=True,
# )